# Проведённые эксперименты

## где мы взяли данные

В экспериментах использована официальная история курсов Банка России из XML API `https://www.cbr.ru/scripts/XML_dynamic.asp`. Период данных зафиксирован с 1 января 2019 года по 2 сентября 2026 года, поэтому результат не сдвигается при появлении новых публикаций. Основные валютные коридоры — `RUB→AMD`, `RUB→KZT`, `RUB→KGS`, `RUB→TJS` и `RUB→UZS`. Для части расширенных экспериментов также загружаются USD, EUR и CNY как контекст общего движения валютного рынка.

Исходный показатель `rub_per_unit` — количество рублей за одну единицу валюты получателя: рубли за один драм, тенге, сом, сомони или сум. Номиналы ЦБ нормализуются, поэтому ряды разных валют имеют одну семантику. Чем `rub_per_unit` ниже, тем выгоднее перевод для отправителя. Горизонт считается не в календарных днях, а в свежих публикациях курса ЦБ. Это не создаёт искусственных наблюдений на выходных. Официальный курс ЦБ является воспроизводимым proxy: он не равен внутридневному курсу исполнения в приложении банка.

## целевая переменная

Основные эксперименты с логистической регрессией и CatBoost решают бинарную задачу `stay_not_worse` на горизонте пяти публикаций. Для даты `T` целевая переменная равна единице, если ни один из следующих пяти курсов не оказался ниже сегодняшнего:

`y(T) = 1[min(q(T+1), …, q(T+5)) ≥ q(T)]`,

где `q(T)` — `rub_per_unit`. Модель выдаёт безразмерную вероятность от 0 до 1: оценку вероятности того, что в следующие пять публикаций более выгодного курса не появится. Точный будущий курс, его изменение в рублях или экономию клиента модель не предсказывает. Вероятность превращается в сигнал после сравнения с порогом. Дополнительно код требует наличия объяснимого факта на дату `T`, чтобы модель не могла сформировать необъяснимый пуш.

Этот target выбран из-за асимметрии ошибки. Ситуация «банк предложил перевести сейчас, после чего курс стал выгоднее» дороже для доверия клиента, чем пропуск хорошего дня. Горизонт `h=5` — рабочее приближение короткого периода, в течение которого клиент реально может отложить перевод. В коде также предусмотрены targets `local_min` — сегодняшний курс является минимумом окна `T−h … T+h` — и `window_closing` — через `h` публикаций курс строго выше сегодняшнего. Это разные продуктовые вопросы. `local_min` использовался в раннем baseline на горизонтах 1, 3, 5, 10 и 20, но не является target основного ML-бэктеста.

Модели получают только информацию, доступную на момент `T`. Группа признаков A содержит четыре бинарных индикатора: momentum, level, reversal и seasonality. Группа B описывает силу этих фактов числами: длину непрерывного снижения, положение курса в 90-публикационном диапазоне, число публикаций после минимума, величину отскока и число дней до праздника. Группа C содержит изменения курса за 1, 3, 5, 10 и 20 публикаций, волатильность и наклон тренда за 20 публикаций. Группа D содержит динамику, положение и волатильность USD, EUR и CNY, а также движение валюты коридора за вычетом движения USD. В основной конфигурации указаны группы A и B; в матричном эксперименте дополнительно проверены A, A+B, A+B+C и A+B+C+D.

## метрики качества

`signal_hit_rate` — доля сигналов, для которых target равен единице. Для основного target это доля сигналов, после которых в следующие пять публикаций более выгодного курса не возникло. `random_hit_rate` — распространённость target среди всех допустимых дат того же коридора и периода. `lift = signal_hit_rate / random_hit_rate`. Значение 1 означает уровень случайного дня, 1,3 — целевое событие после сигнала встречается на 30% чаще, чем в обычный день.

`freq_matched_hit_rate` — более сопоставимый случайный baseline: из периода многократно выбирается столько же случайных дат, сколько сигналов сформировал оцениваемый метод. `lift_freq_matched` делит точность сигнала на эту величину. Это защищает сравнение от различий в частоте методов.

`bps_forward = (mean(q(T+1), …, q(T+5)) − q(T)) / q(T) × 10 000` показывает среднюю выгоду сегодняшнего курса относительно среднего будущего курса. Положительное значение полезно клиенту; 100 базисных пунктов соответствуют примерно 1%. Эта метрика необходима, потому что бинарный target одинаково считает как очень маленькое, так и существенное последующее улучшение курса.

`signals_per_week` измеряет среднее число сигналов в неделю. При подборе порога модель ориентируется на диапазон 0,8–2,5 сигнала в неделю; продуктовый ориентир кейса — примерно 1–2 сообщения. `cluster_share` — доля сигналов, пришедших сразу после другого сигнала. Высокое значение означает, что одно длительное окно порождает серию повторных пушей и до запуска нужен cooldown.

Основной критерий успеха, записанный в плане, — OOT lift не ниже 1,3, нижняя граница 95% доверительного интервала выше 1 и сохранение знака эффекта хотя бы на трёх коридорах и нескольких временных окнах. Bootstrap-доверительные интервалы в текущей итоговой таблице ещё не рассчитаны, поэтому соответствие этому критерию пока не доказано.

## сравнение с правилами

В качестве простых правил используются momentum — курс снижается три публикации подряд; level — курс входит в нижние 10% за последние 90 публикаций; reversal — вчера был минимум за 20 публикаций, а сегодня курс вырос; seasonality — до государственного праздника страны получателя осталось не более семи дней. Каждое правило сразу даёт бинарный сигнал и не обучается. Дополнительный baseline случайно выбирает примерно 1,5 даты в неделю.

Логистическая регрессия и CatBoost решают тот же основной target и оцениваются тем же кодом, что правила. Это делает сравнение корректным: для каждого метода берутся одни и те же временные окна, валютные коридоры и метрики. Порог модели выбирается на validation-периоде по сетке фиксированных вероятностей и квантилей с ограничением частоты. В test-период порог переносится без подстройки.

Ранний baseline, представленный на графиках `baseline_lift_h*.png`, отвечает на другой вопрос. Там правила оцениваются по попаданию в локальный минимум `±h` на всей истории, без walk-forward. На горизонте `h=3` momentum показал lift 3,00–3,32 по пяти коридорам, level — 1,44–1,90, seasonality — 1,02–1,22, reversal — 0. Эти числа показывают концентрацию локальных минимумов, а не качество основного target `stay_not_worse`, поэтому их нельзя напрямую сравнивать с ML-результатами. Ноль у reversal является следствием несовпадения правила и target: разворот срабатывает после минимума.

## схема тестирования

Для ML и сопоставления с правилами используется expanding-window walk-forward. Для каждого фолда обучение содержит только более раннюю историю, следующий год используется как validation для обучения/калибровки и выбора порога, а следующий период — как test. После перехода к новому фолду обучающее окно расширяется. Последние пять публикаций каждой валюты из train удаляются (`purge`), потому что их метки смотрят на пять шагов вперёд и иначе могли бы использовать информацию из validation.

На train обучается LogReg или CatBoost. На validation выбирается порог сигнала. На test модель только применяется, без дообучения и подбора порога по результатам этого периода. Простые правила не обучаются, но оцениваются на тех же test-датах. Метрики считаются отдельно по каждому валютному коридору, а не только в среднем. 2022 год выделен отдельно как аномальный режим. Периоды 2023 года — 31 августа 2025 года объединены в отчёте как `wf_oos`. Финальный `oot` не участвует в предыдущих тестовых фолдах и служит последней отложенной проверкой.

В тестах кода дополнительно проверяются отсутствие пересечений train/validation/test и неизменность уже рассчитанных признаков и сигналов при добавлении будущих данных. Это техническая проверка отсутствия прямой утечки, но она не заменяет проверку статистической устойчивости результата.

## сплиты на обучающую и отложенную выборки

Для фолда 2022 года train содержит данные до 1 января 2021 года, validation — 2021 год, test — 2022 год. Для фолда 2023 года train заканчивается перед 1 января 2022 года, validation — 2022 год, test — 2023 год. Для фолда 2024 года train заканчивается перед 1 января 2023 года, validation — 2023 год, test — 2024 год. Для фолда 2025 года train заканчивается перед 1 января 2024 года, validation — 2024 год, test — с 1 января по 31 августа 2025 года. В каждом train перед обучением дополнительно удаляются последние пять публикаций каждой валюты.

Финальный untouched OOT начинается 1 сентября 2025 года и заканчивается последней доступной датой 2 сентября 2026 года. Для него train содержит данные до 1 сентября 2024 года, validation идёт с 1 сентября 2024 года до 31 августа 2025 года, test — с 1 сентября 2025 года до конца набора. Таким образом, порог для финального OOT выбирается без просмотра OOT-результатов.

Ранние baseline-графики локального минимума используют всю историю сразу и не имеют такого разделения. Поэтому они являются разведочным анализом правил, а не доказательством будущего качества.

## выводы об эффективности правил vs индикаторов

Здесь под «правилами/индикаторами» понимаются четыре ручных бинарных сигнала, а сравниваются они с обучаемыми моделями. Вывод зависит от target. На задаче поиска локального минимума momentum и level выглядели сильно, однако это было in-sample-сравнение на всей истории. После перехода к более продуктовому target `stay_not_worse`, честным временным сплитам и финальному OOT результат изменился.

На объединённых walk-forward test-периодах 2023 — август 2025 года все четыре отдельных правила в среднем по пяти коридорам имели lift ниже 1: momentum 0,849, level 0,688, reversal 0,827, seasonality 0,901. Это означает, что на основном target они не превосходили обычный день устойчиво. Они также часто образуют серии: средний `cluster_share` level около 0,889, seasonality около 0,843. Поэтому использовать одиночное правило как готовую политику пушей нельзя.

На финальном OOT правила тоже не дали устойчивого преимущества по всем коридорам: средний lift momentum равен 0,789, level — 0,933, reversal — 0,640, seasonality — 1,083. У seasonality был сильный результат на TJS, но он не перенёсся одинаково на остальные валюты. Положительные средние базисные пункты отдельных правил не компенсируют lift ниже 1 и нестабильность между коридорами.

Лучшим текущим OOT-вариантом оказался CatBoost на группах A+B. Он сформировал 105 сигналов по пяти коридорам, получил средний hit rate 0,409, средний lift 1,321, frequency-matched lift 1,273 и среднюю выгоду около 60 базисных пунктов. По отдельным коридорам lift составил 1,263 для AMD, 1,389 для KGS, 0,733 для KZT, 1,902 для TJS и 1,317 для UZS. То есть преимущество есть на четырёх коридорах из пяти, но KZT остаётся отрицательным исключением. Средняя частота равна лишь 0,411 сигнала в неделю, ниже продуктового ориентира, а средний `cluster_share` около 0,577 показывает необходимость cooldown.

Этот результат пока нельзя считать доказательством готовности модели. Тот же CatBoost A+B на объединённых walk-forward периодах имел средний lift 0,923, а в отдельно сложном 2022 году — 0,684. Следовательно, хороший финальный OOT не сопровождается устойчивым преимуществом во всех исторических режимах. Логистическая регрессия также не дала стабильного результата: некоторые конфигурации формировали слишком мало сигналов или вообще ни одного, а варианты с достаточной частотой в среднем не превзошли случайный день.

Итог экспериментов такой: ручные индикаторы полезны как понятные клиенту факты и признаки модели, но сами по себе не являются надёжным алгоритмом выбора момента для target «дешевле не станет». CatBoost умеет лучше комбинировать их силу и на финальном OOT показал наиболее перспективный результат, однако текущая устойчивость по времени и коридорам недостаточна для заявления о готовности к запуску. Перед продуктовым пилотом нужны доверительные интервалы, cooldown, настройка приемлемой частоты, проверка на курсе исполнения банка и повторная out-of-time проверка на новых данных.

